# Introduction to LangGraph

In this notebook, we will build a **flexible agent system** using **LangGraph** that integrates multiple tools to handle different types of queries.  
The agents will run on a local **LLM** (such as Ollama’s Llama model) and coordinate external tools to fetch and process information efficiently.

### 🔍 Agents Implemented

1. **Q&A Agent** – Uses a question-answering tool to provide direct answers to user queries.  
2. **Search Agent** – Combines **Wikipedia** and **Arxiv** tools to retrieve detailed information and research content without summarization.

### ⚡ Workflow
- The user submits a query.  
- The system routes the query to the appropriate agent based on the context.  
- The selected agent uses its tools to fetch relevant data.  
- The final result is presented with intermediate reasoning steps visible for debugging and analysis.

This setup demonstrates how **tool-based reasoning + structured workflows** can enhance information retrieval and question-answering systems, offering more accurate and traceable responses compared to a single LLM approach.


## ⚙️ Installations & Setup

Before running the notebook, make sure all required libraries are installed. 

In [ ]:
!apt-get update
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
# ollama serve # In terminal

In [ ]:
!ollama pull llama3.1:8b

In [ ]:
!pip install langchain==0.3.28 langchain-google-genai langchain-community langchain-core numpy==1.26.4 langgraph wikipedia ddgs arxiv

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 45.6 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6048 sha256=72950a97eb46d55ed5ea32482d2b7ce88a3e698970812012564bd90e7d87e91c
  Stored in directory: /tmp/pip-ephem-wheel-cache-lyx8rfcj/wheels/f0/69/93/a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


## 1. Environment Setup & Imports

This block imports the **core libraries and utilities** required to build an **agentic AI system** using **LangChain** and **LangGraph**. Each import plays a specific role in enabling reasoning, tool use, and graph-based workflow execution.

### 🔑 Key Concepts in this Block

- **TypedDict (typing)**  
  Used to define structured dictionaries for type-safe state management in LangGraph workflows.

- **LangChain Core Modules**  
  - `Ollama`: Interface to **local LLMs** served by [Ollama](https://ollama.com), enabling fast, private inference.  
  - `initialize_agent`, `AgentType`, `Tool`, `AgentExecutor`, `create_react_agent`: Utilities for creating **agents** that can reason, plan, and act with external tools.  

- **Community Tools (langchain_community)**  
  - `WikipediaQueryRun`, `ArxivQueryRun`: Prebuilt tools that allow agents to query **Wikipedia** and **Arxiv** for factual knowledge.  
  - `WikipediaAPIWrapper`: Handles lower-level API interactions with Wikipedia.  

- **LangGraph**  
  - `StateGraph`, `END`: Core primitives for defining **graph-based agent workflows**.  
    - `StateGraph` lets you represent an agent as a graph of nodes (steps) and edges (transitions).  
    - `END` marks a terminal state in the workflow.  

- **Prompt Engineering**  
  - `PromptTemplate`: Provides a way to **structure prompts** with variables, ensuring consistency and reusability in LLM interactions.  


This setup ensures we have all the **foundational components** ready:
1. LLM backends (`Ollama`)  
2. Agent-building utilities (LangChain)  
3. External knowledge tools (Wikipedia, Arxiv)  
4. Graph workflow management (LangGraph)  
5. Presentation utilities (IPython display + Markdown)  
6. Prompt engineering (PromptTemplate)  

Together, these imports provide the **building blocks** for creating a **LangGraph-driven multi-tool agent** that can reason, fetch external data, and present results cleanly.

---


In [1]:
from typing import TypedDict, Optional
from langchain.llms import Ollama
from langchain.agents import initialize_agent, AgentType, Tool, AgentExecutor, create_react_agent
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph import StateGraph, END
from typing import TypedDict
from IPython.display import display, Markdown
from langchain.prompts import PromptTemplate

## 2. Creating a Wikipedia Tool

In this block, we define a **Wikipedia search tool** that the agent can call when it needs external factual information.

### 🔑 Key Components

- **`Tool` (LangChain)**  
  A wrapper class that defines how an **LLM agent** can interact with an external function or API.  
  Each tool requires:
  1. `name`: A human-readable identifier the agent uses when reasoning about which tool to call.  
  2. `func`: The callable function that executes the tool’s behavior.  
  3. `description`: A natural language explanation that guides the agent on when to use the tool.  

- **`WikipediaQueryRun` (langchain_community.tools)**  
  A prebuilt LangChain utility that knows how to query **Wikipedia** through the API wrapper and return results in text form.  

- **`WikipediaAPIWrapper` (langchain_community.utilities)**  
  Provides the actual API interaction with **Wikipedia**. It abstracts details such as search requests, fetching summaries, and error handling.  

By creating `wiki_tool`, we equip the agent with the ability to **search Wikipedia dynamically during reasoning**.  
- The `name` ("Wikipedia") will appear in the agent’s reasoning trace.  
- The `description` helps the agent know this tool is best suited for **general or academic knowledge retrieval**.  
- The `func` handles the logic of running a Wikipedia query with the provided search string.  

This modular design allows agents to **plug in multiple tools** (e.g., Arxiv, calculators, custom APIs) and select the most relevant one during execution.

---


In [2]:
wiki_tool = Tool(
    name="Wikipedia",
    func=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper()).run,
    description="Search Wikipedia for academic/general knowledge. Input should be a plain search string.."
)

## 3. Creating an Arxiv Tool

In this block, we define an **Arxiv search tool** that allows the agent to retrieve academic research papers, particularly in technical fields.

### 🔑 Key Components

- **`Tool` (LangChain)**  
  A standard LangChain interface that defines how the agent can use an external resource.  
  - `name`: `"Arxiv"` – the identifier the agent will reference in its reasoning.  
  - `func`: The callable logic, here `ArxivQueryRun().run`, which executes the actual Arxiv search.  
  - `description`: A natural-language guide that informs the agent **when and why** this tool should be chosen.  

- **`ArxivQueryRun` (langchain_community.tools)**  
  Provides the ability to query [arXiv.org](https://arxiv.org), an open-access repository of research papers in physics, math, computer science, and AI/ML.  
  It is particularly useful for **cutting-edge and recent preprints** that may not yet be peer-reviewed or published in journals.  

This tool gives the agent **direct access to research papers**, enabling it to:  
- Retrieve recent findings in fast-moving fields like AI and machine learning.  
- Search topics from the 1990s onward, aligning with arXiv’s archive.  
- Complement Wikipedia by providing **technical and specialized academic content** that may not be covered in general knowledge sources.  

With `arxiv_tool`, the agent can balance **breadth (Wikipedia)** and **depth (Arxiv)** when answering user queries.

---


In [3]:
arxiv_tool = Tool(
    name="Arxiv",
    func=ArxivQueryRun().run,
    description="Search Arxiv for research papers. arXiv is best for cutting-edge preprints (1990s–present) in physics, math, CS, and AI/ML where speed matters.Input should be a plain search string."
)

## 4. Creating a Summarization Tool

This block defines a simple **summarizer tool** that agents can call to condense long pieces of retrieved text into a shorter context.

### 🔑 Key Components

- **`summarize_text` (custom function)**  
  - Input: A string (`text`).  
  - Output: A **truncated summary** consisting of the first 200 characters of the text, prefixed with `"Summary:"`.  
  - Purpose: Acts as a lightweight summarizer for quick context extraction, without invoking an LLM.  

- **`Tool` (LangChain)**  
  Wraps the `summarize_text` function into a format that agents can understand and use.  
  - `name`: `"Summarizer"` – how the agent refers to this capability.  
  - `func`: The callable function (`summarize_text`) that executes the summarization logic.  
  - `description`: Explains that the tool should **always be used to condense retrieved text**, ensuring agents don’t overload context windows with long passages.  

The `summarizer_tool` enhances the agent’s ability to:  
1. **Reduce verbosity** – prevents agents from carrying full-length documents into the reasoning process.  
2. **Improve efficiency** – keeps prompts concise and focused for LLM inference.  
3. **Support downstream tasks** – condensed summaries can then be fed into reasoning chains, stored in memory, or combined with other tool outputs.  

Although this is a **simple truncation-based summarizer**, the modular design means it could later be replaced with a more sophisticated LLM-based summarization tool while keeping the same interface.

---


In [4]:
def summarize_text(text: str) -> str:
    return "Summary: " + text[:200] + "..."  # simple truncation summary

summarizer_tool = Tool(
    name="Summarizer",
    func=summarize_text,
    description="Always condense retrieved text into concise context."
)

## 5. Initializing the LLM (Ollama)

In this block, we set up the **language model backend** that the agent will use for reasoning, summarization, and decision-making.

### 🔑 Key Components

- **`Ollama` (LangChain Integration)**  
  - `Ollama` is a **local model serving framework** that allows you to run LLMs directly on your machine without relying on cloud APIs.  
  - It provides **low-latency inference**, **privacy**, and **fine control** over the model lifecycle.  
  - LangChain’s `Ollama` wrapper lets agents seamlessly interact with locally hosted models as if they were remote LLMs.  

- **Model Selection: `llama3.1:8b`**  
  - Refers to **Meta’s LLaMA 3.1**, a powerful open-weight model.  
  - `8b` indicates the parameter size (**8 billion parameters**), balancing performance with hardware efficiency.  
  - This choice offers strong reasoning capabilities for multi-tool agents without requiring extreme GPU resources.  

By creating `llm = Ollama(model="llama3.1:8b")`, we:  
1. Establish the **core reasoning engine** of the agent.  
2. Enable private, local inference (no external API calls required).  

---


In [5]:
llm = Ollama(model="llama3.1:8b")

/tmp/ipykernel_3184/1421045254.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1:8b")


## 6. Creating a QnA Tool with the LLM 

In this block, we design a **question-answering tool** that allows the agent to directly query the LLM for concise responses, without needing external APIs.

### 🔑 Key Components

- **`llm_qna_tool_func` (custom function)**  
  - Input: A natural language `query`.  
  - Behavior: Wraps the query with an **instructional prompt** —  
    `"Answer the question clearly and concisely:"`  
    before passing it to the LLM (`llm.predict`).  
  - Output: A text response generated by the LLM.  

- **`Tool` (LangChain)**  
  Wraps the function into a LangChain-compatible tool.  
  - `name`: `"QnA_Tool"` — how the agent will reference this tool internally.  
  - `func`: The callable (`llm_qna_tool_func`) that executes the QnA logic.  
  - `description`: Explains that this tool is meant to **directly answer user questions** by leveraging the LLM’s reasoning.  

The `QnA_Tool` is an essential component of the agent’s toolkit:  
1. **Direct Reasoning** – When no external lookup (Wikipedia, Arxiv) is required, the LLM itself can generate an answer.  
2. **Concise Responses** – The function’s prompt ensures outputs remain **focused and clear** rather than verbose.  
3. **Fallback Mechanism** – Provides a baseline QnA ability even if external APIs fail or are not suitable.  

By combining this with external knowledge tools (Wikipedia, Arxiv) and summarization, the agent achieves a **balanced workflow**:  
- **Retrieve** → **Summarize** → **Answer with LLM**.  

---


In [6]:
# QnA Agent = add a simple LLM answering tool
def llm_qna_tool_func(query: str) -> str:
    return llm.predict(f"Answer the question clearly and concisely:\n\n{query}")

llm_qna_tool = Tool(
    name="QnA_Tool",
    func=llm_qna_tool_func,
    description="Answer user queries directly using reasoning."
)

## 7. Defining a Custom Prompt Template for the Agent

In this block, we construct a **structured prompt** that governs how the agent should reason, decide when to use tools, and produce final answers. This follows the **ReAct (Reason + Act)** paradigm, which separates *thoughts* (reasoning) from *actions* (tool use).

### 🔑 Key Components

- **`template` (multiline string)**  
  This text defines the **behavioral rules** for the agent:
  1. **Option 1 – Tool Use**  
     - If the LLM needs more information, it must explicitly choose a tool.  
     - Format:  
       ```
       Action: <tool name>  
       Action Input: <query string>  
       ```  
     - Restrictions:  
       - Wikipedia → factual/encyclopedic queries  
       - Arxiv → academic research (1990s–present) in physics, math, CS, AI/ML  
       - No self-summarization — raw outputs are summarized later.  

  2. **Option 2 – Final Answer**  
     - If enough context is gathered, the agent must clearly state its reasoning and provide a comprehensive final response.  
     - Format:  
       ```
       Thought: I now know the final answer  
       Final Answer: <response>  
       ```  

  3. **Strict Output Rules**  
     - Never mix `Action` and `Final Answer` in the same step.  
     - Always summarize retrieved results before final reasoning.  
     - Up to **3 tool-use cycles** are allowed.  

- **`PromptTemplate.from_template(template)` (LangChain)**  
  Converts the raw string into a **structured prompt object** that can be fed into LangChain’s agent system.  
  - Automatically fills placeholders like `{tools}`, `{tool_names}`, `{input}`, and `{agent_scratchpad}` during runtime.  
  - Ensures consistency across agent invocations.  

This prompt acts as the **control system** for the agent’s reasoning loop:  
- Encourages **step-by-step reasoning** (“Thought”) before actions.  
- Forces clear separation of **decision-making** and **tool usage**.  
- Enforces discipline so the agent doesn’t hallucinate tool outputs or skip summarization.  

By explicitly encoding these rules, we make the agent’s behavior much more **transparent, predictable, and reliable**, aligning with best practices for ReAct-style agents.

---


In [7]:
template = '''
Answer the following questions as best you can. You have access to the following tools:

{tools}

You must follow EXACTLY one of these two options in your output:

---

Option 1 (if you need to use a tool):
Question: the input question you must answer
Thought: reasoning about which tool to use

When you decide to use a tool, output ONLY these lines (nothing else):

Action: <tool name, exactly as given in [{tool_names}]>
Action Input: <input for the tool>

Rules for tool use:
- For factual/encyclopedic queries → use Wikipedia.
- For academic research papers (physics, math, CS, AI/ML, 1990s–present) → use Arxiv.
- Do NOT attempt to summarize yourself; raw results will be summarized later.

(then wait for the Observation. You may repeat this cycle up to 3 times.)

---

Option 2 (if you are ready to give the final response):
Thought: I now know the final answer
Final Answer: <your detailed and comprehensive response>

---

⚠️ Rules:
- NEVER output both an Action and a Final Answer in the same step.
- Always summarize search results before final reasoning.
- Keep the Final Answer detailed and comprehensive.
Begin!

Question: {input}
{agent_scratchpad}
'''

prompt = PromptTemplate.from_template(template)

## 8. Building a Search Agent with Wikipedia + Arxiv

In this block, we assemble a **ReAct-style agent** that can use both **Wikipedia** and **Arxiv** as external knowledge sources, guided by the custom prompt we defined earlier.

### 🔑 Key Components

- **`create_react_agent` (LangChain)**  
  - Constructs an agent that follows the **ReAct (Reason + Act)** framework.  
  - The agent alternates between *thinking* (`Thought`) and *acting* (`Action`) by selecting tools.  
  - Parameters:  
    - `tools=[wiki_tool, arxiv_tool]` → gives the agent access to Wikipedia and Arxiv.  
    - `llm=llm` → the Ollama-powered LLaMA 3.1 (8B) model is the reasoning engine.  
    - `prompt=prompt` → enforces the structured reasoning rules we set earlier.  

- **`AgentExecutor` (LangChain)**  
  - Wraps the agent in an execution engine that handles **running the full reasoning loop**.  
  - Key settings:  
    - `tools=[wiki_tool, arxiv_tool]` → re-confirms which tools are available during execution.  
    - `verbose=True` → prints the agent’s reasoning and tool usage step by step.  
    - `handle_parsing_errors=True` → prevents execution from breaking if the LLM produces slightly malformed outputs.  
    - `return_intermediate_steps=True` → allows inspection of the **thought-action-observation chain**, useful for debugging and transparency.  

The **Search Agent** acts as the **retrieval specialist** within the larger workflow:  
1. Handles **fact-based queries** (Wikipedia).  
2. Handles **research-focused queries** (Arxiv).  
3. Uses the **structured ReAct prompt** to decide when and how to invoke each tool.  
4. Provides **traceable reasoning** through verbose mode and intermediate steps.  

At this stage, the agent is **not yet connected to the summarizer** — it returns raw results from Wikipedia and Arxiv. Summarization will be added later to improve context handling and reduce verbosity.

---


In [8]:
# Search Agent = Wikipedia + Arxiv  (NO summarizer here)
search_agent = create_react_agent(
    tools=[wiki_tool, arxiv_tool],
    llm=llm,
    prompt=prompt
)
search_agent_executor = AgentExecutor(
    agent=search_agent,
    tools=[wiki_tool, arxiv_tool],
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

## 9. Building a QnA Agent with Direct LLM Reasoning

This block defines a **question-answering agent** that relies only on the **LLM QnA tool**, without external retrieval from Wikipedia or Arxiv.

### 🔑 Key Components

- **`create_react_agent` (LangChain)**  
  - Creates a ReAct-style agent where the only available action is the **QnA tool**.  
  - Parameters:  
    - `tools=[llm_qna_tool]` → restricts the agent to using direct LLM reasoning for answers.  
    - `llm=llm` → powered by the local **Ollama LLaMA 3.1 (8B)** model.  
    - `prompt=prompt` → applies the same structured ReAct rules as the search agent, ensuring consistency.  

- **`AgentExecutor` (LangChain)**  
  - Runs the agent and enforces execution logic.  
  - Key settings:  
    - `tools=[llm_qna_tool]` → confirms that only the QnA tool is available.  
    - `verbose=True` → prints intermediate thoughts and actions for transparency.  
    - `handle_parsing_errors=True` → makes the agent more robust to imperfect outputs.  
    - `return_intermediate_steps=True` → allows inspection of the reasoning trace.  

The **QnA Agent** is a **minimalist reasoning agent** that:  
1. Answers questions directly via the LLM, without external knowledge lookups.  
2. Provides a **baseline comparison** against tool-augmented agents (like the Search Agent).  
3. Ensures that the system can still function if external APIs (Wikipedia, Arxiv) are unavailable.  

This design is useful for scenarios where:  
- The user query does not require new information beyond the model’s internal knowledge.  
- Speed and simplicity are more important than research-grade retrieval.  

---


In [9]:
# QnA Agent = only QnA tool
qna_agent = create_react_agent(
    tools=[llm_qna_tool],
    llm=llm,
    prompt=prompt
)
qna_agent_executor = AgentExecutor(
    agent=qna_agent,
    tools=[llm_qna_tool],
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

## 10. Defining the Agent State Schema

In this block, we define a **typed state dictionary** that will track the agent’s progress and information as it moves through the workflow.

### 🔑 Key Components

- **`AgentState` (TypedDict from `typing`)**  
  - A Python feature that allows us to define **structured dictionaries with type annotations**, improving readability, debugging, and IDE support.  
  - Keys in this schema:  
    - `query: str` → the user’s original input question.  
    - `next: Optional[str]` → marker for which step or node should be executed next in the graph.  
    - `context: Optional[str]` → stores **retrieved search results** (e.g., from Wikipedia or Arxiv) that will later be summarized or reasoned over.  
    - `answer: Optional[str]` → the final response produced by the agent after reasoning.  

The `AgentState` provides a **consistent data structure** for passing information between nodes in the LangGraph workflow.  
It ensures that:  
1. The **user’s query** remains accessible across all nodes.  
2. Retrieved **search results** are stored for later summarization.  
3. The **final answer** is tracked and returned cleanly at the end of execution.  

By explicitly defining this state schema, we make the graph-based workflow more **robust, transparent, and type-safe** — a best practice for scaling to more complex multi-step agents.

---


In [10]:
class AgentState(TypedDict):
    query: str
    next: Optional[str]
    context: Optional[str]   # <-- add this for search results
    answer: Optional[str]

## 11. Planner Agent: Routing Queries 

This block introduces a **planner node** - a lightweight decision-making function that routes user queries to the appropriate downstream agent.

### 🔑 Key Components

- **`planner_agent(state: AgentState)`**  
  - Accepts the current agent `state` as input.  
  - Reads the `query` (user’s input question).  
  - Uses simple keyword-based logic to decide the **next node** in the workflow.  

- **Decision Logic**  
  - If the query contains words like `"research"` or `"paper"` → the request likely requires academic sources.  
    - The function returns `{ "next": "search_agent" }`, instructing LangGraph to hand control to the **Search Agent** (Wikipedia + Arxiv).  
  - Otherwise, for general or conversational queries → it routes to the **QnA Agent**.  
    - The function returns `{ "next": "qna_agent" }`.  

The planner acts as a **router or controller** for the workflow.  
It ensures that:  
1. **Research-oriented queries** leverage external tools (Arxiv, Wikipedia).  
2. **General knowledge queries** are answered directly by the LLM, without unnecessary tool usage.  

This mirrors how production-grade agentic systems often work: a **high-level planner** decides *which specialized sub-agent* should handle each query, improving efficiency and reliability.

---


In [11]:
def planner_agent(state: AgentState):
    query = state["query"]
    if "research" in query.lower() or "paper" in query.lower():
        return {"next": "search_agent"}
    else:
        return {"next": "qna_agent"}

## 12. Running the Search Agent

This block defines how the workflow executes the **Search Agent** when a research-oriented query is routed here.

### 🔑 Key Components

- **`run_search_agent(state: AgentState)`**  
  - Accepts the current workflow `state`.  
  - Orchestrates a retrieval + summarization pipeline using the ReAct-based search agent and summarizer.  

- **Execution Steps**  
  1. **Run the Search Agent**  
     - `search_agent_executor.invoke(...)` sends the user’s query to the **ReAct agent** (configured with Wikipedia + Arxiv tools).  
     - This returns a structured response containing raw tool outputs and intermediate reasoning steps.  

  2. **Extract Useful Text**  
     - The response may include nested dictionaries and intermediate traces.  
     - Code checks if `"output"` is present; otherwise, converts the response to a string.  
     - Result: clean raw text from Wikipedia/Arxiv retrieval.  

  3. **Summarize the Results**  
     - Calls `summarizer_tool.func(...)`, which condenses the raw retrieved text into a **concise context**.  
     - This ensures downstream agents don’t get overwhelmed with long, noisy passages.  

  4. **Update State**  
     - Stores the summarized results in `state["context"]`.  
     - Sets `"next": "qna_agent"`, ensuring the workflow continues with the **QnA Agent** to produce the final answer.  

The `run_search_agent` serves as the **retrieval-and-condense stage** of the pipeline.  
It ensures that:  
- Complex queries involving external knowledge are answered using **factual evidence**.  
- The agent workflow stays clean by storing only **summarized context**, not verbose raw results.  
- The next step (QnA Agent) can focus on reasoning over **relevant, digestible information**.  

In short, this function bridges **retrieval (Wikipedia/Arxiv)** and **reasoning (LLM QnA)** in a controlled, structured way.


In [12]:
def run_search_agent(state: AgentState):
    # Run the ReAct search agent
    raw_response = search_agent_executor.invoke({"input": f"{state['query']}"})

    # Extract useful text (strip dicts & intermediate steps)
    if isinstance(raw_response, dict) and "output" in raw_response:
        search_text = raw_response["output"]
    else:
        search_text = str(raw_response)

    # Summarize the search results
    summarized = summarizer_tool.func(search_text)

    # Store only clean summary in context
    return {"context": summarized, "next": "qna_agent"}

## 13. Running the QnA Agent 

This block defines how the **QnA Agent** produces the final answer for the user, optionally leveraging the summarized search results.

### 🔑 Key Components

- **`run_qna_agent(state: AgentState)`**  
  - Accepts the current workflow `state`.  
  - Combines both the **retrieved context** (if available) and the user’s **original query** into a single prompt.  
  - Routes this enriched input into the QnA Agent for final reasoning.  

- **Execution Steps**  
  1. **Build Combined Input**  
     - `combined_input` includes:
       - `"Context:"` → summarized results from the Search Agent (if present).  
       - `"Question:"` → the original user query.  
     - This ensures the QnA Agent reasons over **evidence + question**, not the query alone.  

  2. **Run the QnA Agent**  
     - `qna_agent_executor.invoke(...)` is called with the combined input.  
     - The QnA Agent (powered by the Ollama LLaMA model) uses this to generate a detailed, context-aware response.  

  3. **Extract Final Answer**  
     - The raw response may include extra metadata or intermediate steps.  
     - Code checks for the `"output"` field and extracts only the final answer text.  
     - This is stored in `state["answer"]`.  

The `run_qna_agent` function represents the **final reasoning stage** of the workflow.  
It ensures that:  
- Context from prior retrieval is **integrated** into the final answer.  
- Responses are **concise, evidence-informed, and user-ready**.  
- The workflow cleanly concludes with a single `answer` field.  

Together with the Search Agent, this forms a **retrieval-augmented generation (RAG)-style pipeline** inside a LangGraph workflow — where external knowledge is retrieved, summarized, and then synthesized into a high-quality final answer.

---


In [13]:
def run_qna_agent(state: AgentState):
    combined_input = f"Context:\n{state.get('context','')}\n\nQuestion: {state['query']}"
    response = qna_agent_executor.invoke({"input": combined_input})
    
    # Extract final output only
    if isinstance(response, dict) and "output" in response:
        return {"answer": response["output"]}
    return {"answer": str(response)}

## 14. Building the LangGraph Workflow

This block assembles all the previously defined components into a **graph-based workflow** using LangGraph.  
The graph formalizes how queries move from **planning → retrieval → reasoning → final answer**.

### 🔑 Key Components

- **`StateGraph(AgentState)`**  
  - Creates a graph where each node processes and updates the structured `AgentState`.  
  - This enforces consistency across all nodes in the workflow.  

- **Nodes Added**  
  - `"planner"` → routes queries based on intent.  
  - `"search_agent"` → performs retrieval (Wikipedia + Arxiv) and summarization.  
  - `"qna_agent"` → produces the final context-aware answer.  

- **Conditional Routing**  
  - `route_from_planner` checks `state["next"]` (set by the planner).  
  - Adds **conditional edges**:
    - `"search_agent"` if the query is research-oriented.  
    - `"qna_agent"` if the query is general.  

- **Fixed Edges**  
  - After `"search_agent"` → always flow into `"qna_agent"`.  
  - `"qna_agent"` → connects to `END`, marking workflow completion.  

- **Entry Point**  
  - The workflow always starts at `"planner"`.  
  - Ensures all queries are first analyzed and routed intelligently.  

- **Compile Graph**  
  - `app = graph.compile()` finalizes the workflow into an executable application object.  
  - This object can now process queries end-to-end with LangGraph’s engine.  

This section defines the **agentic pipeline architecture** in LangGraph.  
It transforms our independent functions (planner, search, QnA) into a **cohesive, state-driven workflow**.  

In effect, the graph acts as a **blueprint for agent behavior**:
1. **Planner** decides → where to go.  
2. **Search Agent** gathers evidence (if needed).  
3. **QnA Agent** synthesizes the final answer.  

This modular, declarative structure is what makes LangGraph especially powerful for **reliable, multi-step agent design**.

---


In [14]:
# ----------------------------
# 7) Build LangGraph
# ----------------------------
graph = StateGraph(AgentState)

graph.add_node("planner", planner_agent)
graph.add_node("search_agent", run_search_agent)
graph.add_node("qna_agent", run_qna_agent)

# Conditional routing
def route_from_planner(state: AgentState):
    return state["next"]

graph.add_conditional_edges(
    "planner",
    route_from_planner,
    {
        "search_agent": "search_agent",
        "qna_agent": "qna_agent",
    }
)

# After search → always go to QnA
graph.add_edge("search_agent", "qna_agent")

# QnA ends the flow
graph.add_edge("qna_agent", END)

graph.set_entry_point("planner")

app = graph.compile()


## 15. Visualizing the LangGraph Workflow

In this block, we generate a **graph diagram** of the compiled LangGraph workflow using Mermaid (a lightweight diagramming syntax).  
This helps us **see the flow of execution** across planner, search, and QnA nodes.

### 🔑 Key Components

- **`app.get_graph().draw_mermaid()`**  
  - Extracts the workflow graph structure from the compiled LangGraph `app`.  
  - Converts it into Mermaid syntax so it can be rendered directly in the notebook.  

This step turns our **abstract LangGraph workflow** into a **clear visual diagram**.  

---


In [15]:
custom_styles = """
%%{init: {"theme":"default", "flowchart": {"htmlLabels": true, "nodeSpacing": 50, "rankSpacing": 50, "nodeHeight": 100}} }%%
"""
mermaid_code = app.get_graph().draw_mermaid()
# Inject styles at the top of the mermaid code
mermaid_code = custom_styles + "\n" + mermaid_code
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid

%%{init: {"theme":"default", "flowchart": {"htmlLabels": true, "nodeSpacing": 50, "rankSpacing": 50, "nodeHeight": 100}} }%%

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	search_agent(search_agent)
	qna_agent(qna_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	planner -.-> qna_agent;
	planner -.-> search_agent;
	search_agent --> qna_agent;
	qna_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [16]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "what is atomic number of gold"})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Action: Wikipedia
Action Input: "atomic number of gold"Wikipedia is not a valid tool, try one of [QnA_Tool].---

Option 1 (if you need to use a tool):

Question: what is the atomic number of gold?
Thought: I will use QnA_Tool to answer this question.

Action: QnA_Tool
Action Input: "atomic number of gold"

/tmp/ipykernel_3184/3084972074.py:3: LangChainDeprecationWarning: The method `BaseLLM.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return llm.predict(f"Answer the question clearly and concisely:\n\n{query}")


79.---

Option 2 (if you are ready to give the final response):

Final Answer: The atomic number of gold is 79, which means that an atom of gold has 79 protons in its nucleus. This is a fundamental property of gold as an element and plays a crucial role in its chemical properties and behavior.

> Finished chain.
Final Answer: The atomic number of gold is 79, which means that an atom of gold has 79 protons in its nucleus. This is a fundamental property of gold as an element and plays a crucial role in its chemical properties and behavior.


In [17]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "What is Agentic AI?"})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Option 1
Question: What is Agentic AI?
Thought: Based on the nature of the question, it seems to be a concept-related query that requires specific knowledge about artificial intelligence. Given this, I should use a relevant tool to provide accurate and detailed information.

Action: QnA_Tool
Action Input: "What is Agentic AI?"Agentic AI refers to artificial intelligence that possesses autonomous decision-making capabilities, allowing it to act independently in pursuit of its goals. This means that agentic AI can make decisions without direct human intervention or control, much like a human agent acting on behalf of another entity.

In other words, Agentic AI is self-directed and proactive, capable of adapting and learning from its environment to achieve its objectives, rather than simply executing pre-programmed instructions.---

Option 1
Question: What are the characteristics of Agentic AI?
Thought: Since I need more info

In [18]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "Explain DBSCAN clustering in simple terms."})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Option 1
Question: Explain DBSCAN clustering in simple terms.
Thought: I think I can use the QnA_Tool to provide a general overview of DBSCAN, but I'd like more information. 

Action: QnA_Tool
Action Input: What is DBSCAN and how does it work?**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**

DBSCAN is a widely used unsupervised clustering algorithm that groups data points into clusters based on their density and proximity to each other.

**How it works:**

1. **Core Points**: The algorithm starts by selecting core points, which are points surrounded by at least a minimum number of neighboring points (MinPts). These points have a high enough density to be considered the center of a cluster.
2. **Epsilon (ε)**: DBSCAN also uses an epsilon (ε) value, which is a maximum distance between two points in the same cluster. If two points are within ε of each other, they are considered part of the same cluster

In [19]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "Summarize the architecture of the DETR model."})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Option 1
Question: Summarize the architecture of the DETR model.
Thought: This question seems to be about a specific computer vision model, which is likely related to research in AI/ML. Therefore, I should use the Arxiv tool to find relevant information. 

Action: Arxiv
Action Input: "DETR model architecture"Arxiv is not a valid tool, try one of [QnA_Tool].---

Option 1 (if you need to use a tool):
Question: What does the DETR model architecture entail?
Thought: This question seems to be about a specific computer vision model, which is likely related to research in AI/ML. Therefore, I should use the Arxiv tool to find relevant information.

Action: QnA_Tool
Action Input: "What are the key components and design decisions of the DETR model architecture?"The DETR (DEtection TRansformer) model architecture consists of the following key components and design decisions:

1. **Transformer Encoder**: A standard transformer encoder

In [19]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "Give me a summary of recent research on AI-enabled classrooms."})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Thought: Since this question is about recent research on AI-enabled classrooms, which falls under academic research papers related to education technology, I should use Arxiv to search for relevant papers.

Action: Arxiv
Action Input: Recent research on AI-enabled classroomsPublished: 2025-04-03
Title: A Framework for the Assurance of AI-Enabled Systems
Authors: Ariel S. Kapusta, David Jin, Peter M. Teague, Robert A. Houston, Jonathan B. Elliott, Grace Y. Park, Shelby S. Holdren
Summary: The United States Department of Defense (DOD) looks to accelerate the development and deployment of AI capabilities across a wide spectrum of defense applications to maintain strategic advantages. However, many common features of AI algorithms that make them powerful, such as capacity for learning, large-scale data ingestion, and problem-solving, raise new technical, security, and ethical challenges. These challenges may hinder adoption du

In [21]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "Research papers on diffusion models?"})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Option 1
Thought: The question is about research papers, specifically related to a specific topic (diffusion models). Given that this is an academic research paper query in the fields of physics, math, CS, AI/ML, and it's best for cutting-edge preprints where speed matters, I will use Arxiv.

Action: Arxiv
Action Input: diffusion modelsPublished: 2023-12-06
Title: Generalized Contrastive Divergence: Joint Training of Energy-Based Model and Diffusion Model through Inverse Reinforcement Learning
Authors: Sangwoong Yoon, Dohyun Kwon, Himchan Hwang, Yung-Kyun Noh, Frank C. Park
Summary: We present Generalized Contrastive Divergence (GCD), a novel objective function for training an energy-based model (EBM) and a sampler simultaneously. GCD generalizes Contrastive Divergence (Hinton, 2002), a celebrated algorithm for training EBM, by replacing Markov Chain Monte Carlo (MCMC) distribution with a trainable sampler, such as a diffu

In [22]:
print("\n--- Research Query ---")
res1 = app.invoke({"query": "What is the newest research trend in graph neural networks?"})
print("Final Answer:", res1["answer"])


--- Research Query ---


> Entering new AgentExecutor chain...
Option 1
Thought: The question is asking about the latest trends in graph neural networks, which falls under academic research papers in the field of AI/ML. Therefore, I will use the Arxiv tool to search for recent publications on this topic.

Action: Arxiv
Action Input: "graph neural networks"Published: 2023-06-26
Title: The Deep Arbitrary Polynomial Chaos Neural Network or how Deep Artificial Neural Networks could benefit from Data-Driven Homogeneous Chaos Theory
Authors: Sergey Oladyshkin, Timothy Praditia, Ilja Kröker, Farid Mohammadi, Wolfgang Nowak, Sebastian Otte
Summary: Artificial Intelligence and Machine learning have been widely used in various fields of mathematical computing, physical modeling, computational science, communication science, and stochastic analysis. Approaches based on Deep Artificial Neural Networks (DANN) are very popular in our days. Depending on the learning task, the exact form of DANNs is 